<a href="https://colab.research.google.com/github/Sharafatnoa/DailyPractice/blob/main/2026-10-03-statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Practice — 2026-10-03 — Statistics: Is the Candidate Model *Actually* Better, or Just Lucky on This Split?

**Dataset:** [Palmer Station Penguins](https://github.com/mwaskom/seaborn-data) (344 real penguin
measurements collected at Palmer Station, Antarctica — bill length/depth, flipper length, body
mass, species, island, sex), loaded live from seaborn's public dataset mirror.

## Problem statement

A teammate swaps your team's simple 1-nearest-neighbor species classifier for a Random Forest and
reports "it's more accurate on the test set." In ML quality work this claim is exactly the kind of
thing you're paid to be skeptical of: with a single train/test split, *any* two models will differ
by some amount just from which rows happened to land in the test set. The question that matters
isn't "is the number bigger?" — it's "would this difference show up again on a fresh sample, or is
it noise?" That's a statistical inference question, not a modelling one, and it's one of the most
common things an ML quality engineer is asked to adjudicate before a model ships.

To make the comparison genuinely uncertain (not a blowout either way), the exercise deliberately
restricts both models to two weak, overlapping features (`bill_depth_mm` and `body_mass_g`) instead
of all four measurements — with the full feature set the species separate almost perfectly and
there's nothing interesting left to test.

## What you should produce

Implement three statistical inference functions and use them to answer, with numbers, whether the
Random Forest candidate is significantly better than the 1-NN baseline:

1. `bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, random_state=0)` — a bootstrap
   point estimate and `(1 - alpha)` confidence interval for a single model's metric.
2. `paired_bootstrap_diff(y_true, pred_a, pred_b, metric_fn, n_boot=2000, alpha=0.05, random_state=0)`
   — a confidence interval for the *difference* in metric between two models, resampling both
   models' predictions on the *same* bootstrap indices each draw (paired, not independent).
3. `paired_permutation_test(y_true, pred_a, pred_b, metric_fn, n_perm=2000, random_state=0)` — a
   p-value for the null hypothesis "the two models are equally correct on each test row, and the
   observed difference is due to chance," by randomly swapping each row's two per-model correctness
   labels and recomputing the metric gap.

Then run all three against **both accuracy and macro-F1**, and write 3-5 sentences stating: is the
difference statistically significant at `alpha = 0.05`? Does the answer change depending on which
metric you pick? Would you sign off on shipping the candidate?


## Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
df = df.dropna(subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]).reset_index(drop=True)

print("shape:", df.shape)
print("species counts:\n", df["species"].value_counts())
df.head()


## Baseline vs. candidate

Deliberately weak feature set (`bill_depth_mm`, `body_mass_g` only) so the two models land close
enough together that "is this real?" is a genuine question, not a formality.


In [ ]:
features = ["bill_depth_mm", "body_mass_g"]
X = df[features]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=7
)

# Baseline: already shipped, dead simple.
baseline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))

# Candidate: the teammate's proposed replacement.
candidate = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0)

baseline.fit(X_train, y_train)
candidate.fit(X_train, y_train)

pred_a = baseline.predict(X_test).astype(str)
pred_b = candidate.predict(X_test).astype(str)
y_test_arr = y_test.to_numpy().astype(str)

print("baseline  accuracy:", accuracy_score(y_test_arr, pred_a), " macro-F1:", f1_score(y_test_arr, pred_a, average="macro"))
print("candidate accuracy:", accuracy_score(y_test_arr, pred_b), " macro-F1:", f1_score(y_test_arr, pred_b, average="macro"))


## Your turn: implement the three inference functions below

Notice that accuracy and macro-F1 above don't even agree on which model looks better — that's
your first hint that eyeballing a single point estimate isn't enough. Fill in the `TODO`s.


In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, random_state=0):
    """
    TODO: Bootstrap confidence interval for metric_fn(y_true, y_pred).

    - Draw `n_boot` resamples of the row indices, WITH replacement, each of
      size n = len(y_true).
    - For each resample, recompute metric_fn on the resampled y_true/y_pred.
    - Return (point_estimate, ci_lower, ci_upper), where point_estimate is
      metric_fn evaluated on the *original* (non-resampled) data, and the CI
      bounds are the alpha/2 and 1 - alpha/2 percentiles of the bootstrap
      distribution.
    - Use np.random.default_rng(random_state) for reproducibility.
    """
    raise NotImplementedError


def paired_bootstrap_diff(y_true, pred_a, pred_b, metric_fn, n_boot=2000, alpha=0.05, random_state=0):
    """
    TODO: Bootstrap CI for the *difference* metric_fn(y_true, pred_a) - metric_fn(y_true, pred_b).

    - This must be PAIRED: each bootstrap draw uses the same resampled row
      indices for both pred_a and pred_b, since they're predictions on the
      same test rows. Resampling each model independently would throw away
      the correlation between them and overstate the uncertainty.
    - Return (point_estimate, ci_lower, ci_upper) for the difference.
    """
    raise NotImplementedError


def paired_permutation_test(y_true, pred_a, pred_b, metric_fn, n_perm=2000, random_state=0):
    """
    TODO: Paired permutation test for H0: "model A and model B are equally
    likely to be correct on each test row" (using accuracy-style per-row
    correctness, not metric_fn directly — macro-F1 isn't decomposable
    per-row, but per-row correctness is a reasonable statistic for either).

    - observed statistic = mean(correct_a) - mean(correct_b)
    - For each of n_perm permutations: for every row, independently and
      randomly swap which model's correctness label is "A" and which is "B"
      (a coin flip per row), then recompute the statistic on the shuffled
      labels.
    - p-value = (1 + count of |permuted statistic| >= |observed statistic|)
      / (1 + n_perm)   [the "+1" avoids a p-value of exactly 0]
    - Return (observed_statistic, p_value).
    """
    raise NotImplementedError


In [ ]:
# TODO: put it together.
# 1. For accuracy AND macro-F1 (use f1_score(..., average="macro") as metric_fn):
#      - bootstrap_ci for the baseline
#      - bootstrap_ci for the candidate
#      - paired_bootstrap_diff between them
#    Print each point estimate with its 95% CI.
# 2. Run paired_permutation_test on accuracy and report the p-value.
# 3. In a markdown cell (or comment) below, answer in 3-5 sentences:
#    - Is the accuracy difference significant at alpha=0.05? Does the CI for
#      the difference exclude 0? Does the permutation p-value agree?
#    - Does macro-F1 tell the same story as accuracy, or a different one?
#    - Would you sign off on shipping the candidate? What would you tell the
#      teammate who only looked at the raw accuracy numbers?


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

### Implementation

```python
def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=2000, alpha=0.05, random_state=0):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    point = metric_fn(y_true, y_pred)
    boot = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot[i] = metric_fn(y_true[idx], y_pred[idx])
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lo, hi


def paired_bootstrap_diff(y_true, pred_a, pred_b, metric_fn, n_boot=2000, alpha=0.05, random_state=0):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    n = len(y_true)
    point = metric_fn(y_true, pred_a) - metric_fn(y_true, pred_b)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)  # same indices for both models
        diffs[i] = metric_fn(y_true[idx], pred_a[idx]) - metric_fn(y_true[idx], pred_b[idx])
    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point, lo, hi


def paired_permutation_test(y_true, pred_a, pred_b, metric_fn, n_perm=2000, random_state=0):
    rng = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    correct_a = (pred_a == y_true).astype(float)
    correct_b = (pred_b == y_true).astype(float)
    observed = correct_a.mean() - correct_b.mean()
    n = len(y_true)
    count = 0
    for _ in range(n_perm):
        swap = rng.integers(0, 2, n).astype(bool)
        a = np.where(swap, correct_b, correct_a)
        b = np.where(swap, correct_a, correct_b)
        stat = a.mean() - b.mean()
        if abs(stat) >= abs(observed):
            count += 1
    p_value = (count + 1) / (n_perm + 1)
    return observed, p_value
```

Applying this to the baseline (1-NN) vs. candidate (Random Forest) on the held-out test set:

```python
for name, fn in [("accuracy", accuracy_score),
                  ("macro-F1", lambda yt, yp: f1_score(yt, yp, average="macro"))]:
    a_pt, a_lo, a_hi = bootstrap_ci(y_test_arr, pred_a, fn)
    b_pt, b_lo, b_hi = bootstrap_ci(y_test_arr, pred_b, fn)
    d_pt, d_lo, d_hi = paired_bootstrap_diff(y_test_arr, pred_a, pred_b, fn)
    print(f"--- {name} ---")
    print(f"baseline  {a_pt:.3f}  95% CI [{a_lo:.3f}, {a_hi:.3f}]")
    print(f"candidate {b_pt:.3f}  95% CI [{b_lo:.3f}, {b_hi:.3f}]")
    print(f"diff (baseline - candidate) {d_pt:.3f}  95% CI [{d_lo:.3f}, {d_hi:.3f}]")

obs, p = paired_permutation_test(y_test_arr, pred_a, pred_b, accuracy_score)
print("permutation test on accuracy — observed diff:", obs, " p-value:", p)
```

Representative output on this split (`random_state=7`, `test_size=0.4`):

```
--- accuracy ---
baseline  0.686  95% CI [0.613, 0.766]
candidate 0.774  95% CI [0.701, 0.847]
diff (baseline - candidate) -0.088  95% CI [-0.168, -0.007]
--- macro-F1 ---
baseline  0.640  95% CI [0.571, 0.708]
candidate 0.595  95% CI [0.567, 0.618]
diff (baseline - candidate) 0.045  95% CI [-0.028, 0.118]
permutation test on accuracy — observed diff: -0.088  p-value: 0.047
```

### Interpretation

On **accuracy**, the candidate really does look better: the 95% CI for the difference excludes 0
(barely — the upper bound is -0.007), and the permutation p-value (≈0.047) is just under the
conventional 0.05 threshold. On **macro-F1**, though, the story flips — the baseline's point
estimate is *higher*, and the CI for that difference comfortably includes 0, so there's no
significant evidence either way. This is exactly the kind of result that should make an ML quality
engineer pause rather than sign off: accuracy improved, but only because the candidate is better at
the majority classes, while macro-F1 (which weights every species equally) shows no reliable gain
and even a slight edge to the simpler baseline. Given that the accuracy result is also right at the
significance boundary — and both results come from a single train/test split rather than repeated
resampling of the *data itself* (only the predictions are being bootstrapped here) — the honest
answer is "not yet": ask for cross-validated numbers, check which metric actually matters for the
product (class-weighted accuracy or macro-F1 depending on whether rare species matter), and don't
ship on the strength of one accuracy delta that disappears under a different scoring function.

</details>
